In [ ]:
# Utilities
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from src.data_loader import *
from src.feature_analysis import *
from src.modeling import *
from src.transfer_learning import *
from src.visualization import *
import src.config

sns.set_theme(style="whitegrid")

In [ ]:
# Load datasets
htru1 = load_htru1()
htru2 = load_htru2()

## HTRU1 Exploratory Data Analysis:

In [ ]:
htru1.head()

In [ ]:
def dataset_summary(df, target="class"):
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\nClass Distribution:")
    print(df[target].value_counts(normalize=True))

    return df.describe()

In [ ]:
dataset_summary(htru1, target="class")

In [ ]:
# HTRU1 feature distributions
plot_feature_distributions(htru1, target="class")

In [ ]:
# HTRU1 feature correlations with class label
plot_target_correlations(htru1, target="class")

In [ ]:
# HTRU1 feature correlation heatmap
plot_correlation_heatmap(htru1, target="class")

In [ ]:
# HTRU1 feature importances, correlations, and VIF summary
summary_h1 = feature_summary(
    htru1,
    model_type="xgboost"
)


print(summary_h1)

summary_h1.to_csv(
    "results/tables/htru1_feature_summary.csv"
)

## HTRU2 Exploratory Data Analysis:

In [ ]:
htru2.head()

In [ ]:
dataset_summary(htru2, target="class")

In [ ]:
# HTRU2 feature distributions
plot_feature_distributions(htru2, target="class")

In [ ]:
# HTRU2 feature correlations with class label
plot_target_correlations(htru2, target="class")

In [ ]:
# HTRU2 feature correlation heatmap
plot_correlation_heatmap(htru2, target="class")

In [ ]:
# HTRU2 feature importances, correlations, and VIF summary
summary_h2 = feature_summary(
    htru2,
    model_type="xgboost"
)


print(summary_h2)

summary_h2.to_csv(
    "results/tables/htru2_feature_summary.csv"
)

## Feature Importance Agreement Between HTRU1 and HTRU2 models:

In [ ]:
# Plotting feature importance agreement between HTRU 1 and HTRU 2 models
comparison["Rank_H1"] = (
    summary_h1["Importance_Rank"]
)

comparison["Rank_H2"] = (
    summary_h2["Importance_Rank"]
)

plt.figure(figsize=(8,6))

plt.scatter(
    comparison["Rank_H1"],
    comparison["Rank_H2"]
)

for feature in comparison.index:

    plt.annotate(
        feature,
        (
            comparison.loc[feature,"Rank_H1"],
            comparison.loc[feature,"Rank_H2"]
        )
    )

plt.xlabel("HTRU1 Importance Rank")
plt.ylabel("HTRU2 Importance Rank")

plt.title(
    "Feature Importance Rank Agreement"
)

plt.savefig("results/figures/feature_rank_agreement.png")

plt.show()

## Within-Dataset Training and Testing:

In [ ]:
# Train and test data from same dataset
model_h1, metrics_h1 = (
    within_dataset_experiment(htru1, model_type="xgboost")
)

model_h2, metrics_h2 = (
    within_dataset_experiment(htru2, model_type="xgboost")
)

In [ ]:
print("Performance for within-dataset training:\n")
results_within = pd.DataFrame(
    [metrics_h1, metrics_h2],
    index=["HTRU1","HTRU2"]
)

print(results_within.round(4))

## Cross-Dataset Generalization:

In [ ]:
# Performance results for Training on HTRU1 -> Testing on HTRU2
# Multiple feature sets are trained and tested
results_12 = (
    run_transfer_feature_experiments(
        htru1,
        htru2,
        summary_h1,
        "HTRU1",
        "HTRU2",
        target="class",
        top_k=5,
        vif_threshold=10,
        model_type="xgboost"
    )
)

print(results_12)


results_12.to_csv(
    "results/tables/htru12_transfer_performance.csv"
)

In [ ]:
# Performance results for Training on HTRU2 -> Testing on HTRU1
# Multiple feature sets are trained and tested
results_21 = (
    run_transfer_feature_experiments(
        htru2,
        htru1,
        summary_h2,
        "HTRU2",
        "HTRU1",
        target="class",
        top_k=5,
        vif_threshold=10,
        model_type="xgboost"
    )
)

print(results_21)


results_21.to_csv(
    "results/tables/htru21_transfer_performance.csv"
)

In [ ]:
# Plot cross-dataset experiment results
plot_transfer_metrics(
    results_12,
    "HTRU1 → HTRU2 Transfer Performance",
    save_path="results/figures/htru1_to_htru2.png",
    show=True
)

In [ ]:
plot_transfer_metrics(
    results_21,
    "HTRU2 → HTRU1 Transfer Performance",
    save_path="results/figures/htru2_to_htru1.png",
    show=True
)